# 02 — Sentinel-2: per-acquisition point extraction (raw, no cloud mask)

Samples `COPERNICUS/S2_SR_HARMONIZED` at the labelled points, every acquisition, no masking. Keeps `SCL`, `MSK_CLDPRB`, `MSK_SNWPRB` for masking after download (`QA60`/`MSK_CLASSI_*` excluded — empty for most of this window). Spectral scaled to reflectance; QA bands raw.

Output: `02_RawTimeSeries/S2/{RUN_ID}_long.parquet`.

## Setup

In [2]:
!pip -q install earthengine-api geemap geopandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 40.7 MB/s eta 0:00:00


In [3]:
import os, glob, json, time
import numpy as np
import pandas as pd
import geopandas as gpd
import ee

from google.colab import drive
drive.mount('/content/drive')

ROOT       = '/content/drive/MyDrive/Crop_Classification'
DIR_POINTS = f'{ROOT}/01_Points'
DIR_RAW_TS = f'{ROOT}/02_RawTimeSeries'
DIR_QA     = f'{ROOT}/05_QA'
os.makedirs(f'{DIR_RAW_TS}/S2', exist_ok=True)
os.makedirs(DIR_QA, exist_ok=True)

# Your actual points file (optional cross-check only; the EE asset is the source of truth).
POINTS_GPKG = f'{DIR_POINTS}/wbcrop_points_extended.gpkg'

Mounted at /content/drive


## Configuration

In [4]:
EE_PROJECT = 'ee-geographymanas'

START_DATE = '2023-01-01'
END_DATE   = '2024-12-31'          # end is EXCLUSIVE in filterDate

# Spectral bands (Earth Engine names) — kept exactly as the original notebook.
SPECTRAL_EE  = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
SPECTRAL_OUT = ['B02','B03','B04','B05','B06','B07','B08','B8A','B11','B12']

# QA bands kept RAW so you can mask yourself after download.
# SCL, MSK_CLDPRB, MSK_SNWPRB are populated across the WHOLE 2023-2024 window.
# QA60 and the MSK_CLASSI_* bands are DELIBERATELY EXCLUDED: in this collection QA60 is
# empty 2022-01-25..2024-02-28 and MSK_CLASSI_* are empty before Feb 2024 -- blank for most
# of date window, and a fully-masked band can even drop rows from sampleRegions.
QA_BANDS = ['SCL', 'MSK_CLDPRB', 'MSK_SNWPRB']

# NO cloud masking. Optional loose scene filter: 100 = keep every scene (fully raw).
# Lower it (e.g. 95) only to trim near-useless all-cloud scenes and cut export size.
MAX_SCENE_CLOUD = 100

SCALE_M = 10
DIVIDE_REFLECTANCE = True     # spectral / 10000 -> reflectance 0-1. False = raw integer DN.

EXPORT_FOLDER = 'CropClass_S2_raw'
RUN_ID = f"s2raw_{pd.Timestamp(START_DATE):%Y%m}_{pd.Timestamp(END_DATE):%Y%m}"
LONG_PARQUET = f'{DIR_RAW_TS}/S2/{RUN_ID}_long.parquet'

# CSV asset chosen over the shapefile asset: full field names preserved, integer id,
# identical geometry for point sampling.
POINTS_ASSET = 'projects/ee-geographymanas/assets/WestBengal/wbcrop_points_extended'

MAX_POINTS_PER_TASK = 6000    # districts above this are split by date

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("RUN_ID:", RUN_ID)

RUN_ID: s2raw_202301_202412


## Points

Batch export against the uploaded EE asset (authoritative); local GeoPackage is cross-check only.

In [5]:
fc      = ee.FeatureCollection(POINTS_ASSET)
n_asset = fc.size().getInfo()
props   = fc.first().propertyNames().getInfo()
print(f"Asset points : {n_asset:,}")
print("Properties   :", props)

assert 'id' in props,       "asset has no 'id' field — cannot key the extraction"
assert 'district' in props, "asset has no 'district' field — chunking needs it"

n_unique_id = fc.aggregate_count_distinct('id').getInfo()
print(f"Unique ids   : {n_unique_id:,}  "
      f"({'OK' if n_unique_id == n_asset else '*** DUPLICATE ids — fix before running'})")

# Optional local cross-check (safe to skip if the file isn't in Drive).
try:
    pts = gpd.read_file(POINTS_GPKG, layer='wbcrop_points_extended')
    print(f"\nLocal gpkg   : {len(pts):,} rows | crops: {pts['crop'].nunique()} | "
          f"districts: {pts['district'].nunique()}")
    if len(pts) != n_asset:
        print(f"*** MISMATCH: local {len(pts):,} vs asset {n_asset:,} — re-check the upload.")
except Exception as e:
    pts = None
    print("\nLocal gpkg not read (fine — asset is the source of truth):", repr(e))

Asset points : 74,351
Properties   : ['collection_date', 'harvest', 'sowing', 'district', 'season', 'id', 'state', 'source', 'pheno_stage', 'crop', 'system:index']
Unique ids   : 74,351  (OK)

Local gpkg   : 74,351 rows | crops: 18 | districts: 21


## Build the raw collection

No masking. Spectral bands are scaled to reflectance; SCL / MSK_CLDPRB / MSK_SNWPRB are carried
through unscaled so you can mask after download. `MGRS_TILE` is copied so you can tell apart
the two granules that observe a point on a tile boundary the same day.

In [6]:
OUT_BANDS = SPECTRAL_EE + QA_BANDS

def prep(img):
    """Scale spectral to reflectance; keep QA bands raw. NO cloud masking."""
    spec = img.select(SPECTRAL_EE)
    if DIVIDE_REFLECTANCE:
        spec = spec.divide(10000)
    qa = img.select(QA_BANDS)
    return (spec.addBands(qa)
            .copyProperties(img, ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE', 'MGRS_TILE']))

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterDate(START_DATE, END_DATE)
if MAX_SCENE_CLOUD < 100:
    s2 = s2.filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', MAX_SCENE_CLOUD))
    print(f"Scene filter : keeping CLOUDY_PIXEL_PERCENTAGE <= {MAX_SCENE_CLOUD}")
else:
    print("Scene filter : OFF — every scene kept (fully raw).")

# filterBounds/map are applied per district inside sample_chunk, NOT here: EE's .map() runs
# on every image in the collection even where the result is empty, so filtering to the whole
# state and mapping once per district would make each district process every state-wide scene.
n_img = s2.filterBounds(fc.geometry().bounds()).size().getInfo()
print(f"Scenes over the whole AOI: {n_img:,}")
print("A single district touches only a small fraction of these.")

Scene filter : OFF — every scene kept (fully raw).
Scenes over the whole AOI: 8,104
A single district touches only a small fraction of these.


## Sample

Chunked by district — memory limits and failure isolation.

In [7]:
def sample_chunk(sub, d0, d1):
    """One row per point per acquisition (NO masking), for a point subset and date range.

    The collection is filtered to the subset's bounding box FIRST, so Earth Engine only
    touches scenes that actually cover these points, then prep() runs on that small set.
    """
    region = sub.geometry().bounds()
    col = s2.filterBounds(region).filterDate(d0, d1).map(prep)

    def per_image(img):
        date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        tile = ee.String(img.get('MGRS_TILE'))
        return (img.select(OUT_BANDS)
                .sampleRegions(collection=sub, properties=['id'],
                               scale=SCALE_M, geometries=False, tileScale=16)
                .map(lambda f: f.set('date', date).set('tile', tile)))

    return col.map(per_image).flatten()


def plan_tasks(districts, counts):
    """Split any district above MAX_POINTS_PER_TASK into two halves by date.

    Cost scales with images x points, and the image side is the larger factor, so splitting
    by date (halving images) is more effective than splitting into more districts.
    """
    plan = []
    for d in districts:
        if counts.get(d, 0) > MAX_POINTS_PER_TASK:
            mid = (pd.Timestamp(START_DATE) +
                   (pd.Timestamp(END_DATE) - pd.Timestamp(START_DATE)) / 2).strftime('%Y-%m-%d')
            plan.append((d, START_DATE, mid, 'H1'))
            plan.append((d, mid, END_DATE, 'H2'))
        else:
            plan.append((d, START_DATE, END_DATE, 'all'))
    return plan

In [9]:
tasks = []

# District names come from the ASSET, not the local table: a difference in case or spacing
# would make ee.Filter.eq() return an empty set — the task would run, succeed, and write a
# CSV with zero rows, with no error anywhere.
hist   = fc.aggregate_histogram('district').getInfo()
counts = {k: int(v) for k, v in hist.items() if k}
districts = sorted(counts)

print(f"Districts in the asset: {len(districts)}")
print(f"Points accounted for  : {sum(counts.values()):,} of {n_asset:,}")

if pts is not None:
    local = set(pts['district'].dropna().unique())
    only_asset, only_local = set(districts) - local, local - set(districts)
    if only_asset or only_local:
        print(f"\n*** Name mismatch — asset only: {sorted(only_asset)[:5]}")
        print(f"                   local only: {sorted(only_local)[:5]}")
        print("    Using the asset names, which is what the filter will see.")

plan = plan_tasks(districts, counts)
big  = [d for d in districts if counts[d] > MAX_POINTS_PER_TASK]
print(f"\n{len(districts)} districts -> {len(plan)} tasks")
if big:
    print(f"Split by date (>{MAX_POINTS_PER_TASK:,} points): {big}")

# Show how much the per-district bounds filter saves, on one district.
_d   = plan[0][0]
_sub = fc.filter(ee.Filter.eq('district', _d))
_n   = s2.filterBounds(_sub.geometry().bounds()).size().getInfo()
print(f"\nExample: '{_d}' touches {_n:,} scenes, not {n_img:,} "
      f"({n_img/max(_n,1):.0f}x less work per task)\n")

for d, d0, d1, tag in plan:
    sub  = fc.filter(ee.Filter.eq('district', d))
    name = f"S2raw_{d.replace(' ', '_')}_{tag}"
    t = ee.batch.Export.table.toDrive(
        collection=sample_chunk(sub, d0, d1),
        description=name, folder=EXPORT_FOLDER, fileFormat='CSV')
    t.start()
    tasks.append((name, t))
    print(f"  started: {name:<36} {counts[d]:>6,} points  {d0} -> {d1}")

planned = sum(counts[d] for d, _, _, tag in plan if tag in ('all', 'H1'))
assert planned == sum(counts.values()), (
    f"task plan covers {planned:,} points but the asset has {sum(counts.values()):,}")
print(f"\nEvery one of the {sum(counts.values()):,} points is covered by a task.")
print(f"{len(tasks)} tasks submitted. Monitor: https://code.earthengine.google.com/tasks")
print("Close the notebook if you like — EE keeps running and writes straight to Drive.")

Districts in the asset: 21
Points accounted for  : 74,351 of 74,351

21 districts -> 25 tasks
Split by date (>6,000 points): ['murshidabad', 'nadia', 'pashchim medinipur', 'purba medinipur']

Example: 'alipurduar' touches 290 scenes, not 8,104 (28x less work per task)

  started: S2raw_alipurduar_all                    831 points  2023-01-01 -> 2024-12-31
  started: S2raw_bankura_all                     2,617 points  2023-01-01 -> 2024-12-31
  started: S2raw_birbhum_all                     4,265 points  2023-01-01 -> 2024-12-31
  started: S2raw_dakshin_dinajpur_all              723 points  2023-01-01 -> 2024-12-31
  started: S2raw_darjiling_all                   4,435 points  2023-01-01 -> 2024-12-31
  started: S2raw_haora_all                       1,447 points  2023-01-01 -> 2024-12-31
  started: S2raw_hugli_all                       1,524 points  2023-01-01 -> 2024-12-31
  started: S2raw_jalpaiguri_all                  2,072 points  2023-01-01 -> 2024-12-31
  started: S2raw_jhargram_

In [11]:
for name, t in tasks:
    print(f"{name:<38} {t.status()['state']}")

S2raw_alipurduar_all                   CANCEL_REQUESTED
S2raw_bankura_all                      CANCEL_REQUESTED
S2raw_birbhum_all                      CANCEL_REQUESTED
S2raw_dakshin_dinajpur_all             CANCELLED
S2raw_darjiling_all                    CANCELLED
S2raw_haora_all                        CANCELLED
S2raw_hugli_all                        CANCELLED
S2raw_jalpaiguri_all                   CANCELLED
S2raw_jhargram_all                     CANCELLED
S2raw_koch_bihar_all                   CANCELLED
S2raw_maldah_all                       CANCELLED
S2raw_murshidabad_H1                   CANCELLED
S2raw_murshidabad_H2                   CANCELLED
S2raw_nadia_H1                         CANCELLED
S2raw_nadia_H2                         CANCELLED
S2raw_north_24_parganas_all            CANCELLED
S2raw_paschim_barddhaman_all           CANCELLED
S2raw_pashchim_medinipur_H1            CANCELLED
S2raw_pashchim_medinipur_H2            CANCELLED
S2raw_purba_barddhaman_all             CANCELLED

## Collect the exports

Run once every task shows COMPLETED.

In [12]:
csv_dir = f'/content/drive/MyDrive/{EXPORT_FOLDER}'
files   = sorted(glob.glob(f'{csv_dir}/S2raw_*.csv'))
print(f"CSV files: {len(files)}")

if files:
    ts = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    ts = ts.drop(columns=['system:index', '.geo'], errors='ignore')
    ts = ts.rename(columns=dict(zip(SPECTRAL_EE, SPECTRAL_OUT)))
    ts['date'] = pd.to_datetime(ts['date'])
    print("Shape  :", ts.shape)
    print("Columns:", list(ts.columns))

CSV files: 25
Shape  : (12280147, 16)
Columns: ['B11', 'B12', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'MSK_CLDPRB', 'MSK_SNWPRB', 'SCL', 'date', 'id', 'tile']


## Verify

In [13]:
# Overlapping granules: one point can be observed twice on a day, once per tile. These are
# genuine separate observations in RAW data, so they are KEPT and distinguished by 'tile'.
dup = ts.groupby(['id', 'date']).size()
n_multi = int((dup > 1).sum())
print(f"Point-date pairs with >1 observation (overlapping tiles): {n_multi:,} / {len(dup):,}")
print("Kept as-is; use the 'tile' column to tell them apart or de-duplicate later if you wish.")

valid = ts[SPECTRAL_OUT].notna().all(axis=1)
obs   = ts.groupby('id').size()
print(f"\nRows total            : {len(ts):,}")
print(f"Rows with all spectral : {int(valid.sum()):,} ({valid.mean():.1%})")
print(f"Unique points returned : {ts['id'].nunique():,} of {n_asset:,}")
print(f"Unique dates           : {ts['date'].nunique()}")
print(f"Obs per point (raw)    : min {obs.min()}, median {obs.median():.0f}, max {obs.max()}")

Point-date pairs with >1 observation (overlapping tiles): 1,296,589 / 10,831,218
Kept as-is; use the 'tile' column to tell them apart or de-duplicate later if you wish.

Rows total            : 12,280,147
Rows with all spectral : 12,280,147 (100.0%)
Unique points returned : 74,351 of 74,351
Unique dates           : 289
Obs per point (raw)    : min 142, median 146, max 1169


In [14]:
scl_names = {0:'nodata',1:'saturated',2:'dark',3:'shadow',4:'vegetation',5:'bare',
             6:'water',7:'unclassified',8:'cloud_med',9:'cloud_high',10:'cirrus',11:'snow'}
comp = ts['SCL'].value_counts(normalize=True).sort_index()
print("SCL composition across ALL raw observations (nothing dropped):")
for k, v in comp.items():
    print(f"  {int(k):>2} {scl_names.get(int(k), '?'):<14} {v:>7.2%}")
print("\nCloud/shadow/cirrus classes ARE present here — that is expected for raw data.")
print("Mask on these (and/or MSK_CLDPRB) after download, per your own method.")

SCL composition across ALL raw observations (nothing dropped):
   2 dark             0.00%
   3 shadow           1.42%
   4 vegetation      20.89%
   5 bare            31.14%
   6 water            0.80%
   7 unclassified     1.47%
   8 cloud_med       16.60%
   9 cloud_high      22.71%
  10 cirrus           4.96%

Cloud/shadow/cirrus classes ARE present here — that is expected for raw data.
Mask on these (and/or MSK_CLDPRB) after download, per your own method.


## Monthly coverage (all observations)

In [15]:
ts['month'] = ts['date'].dt.to_period('M')
cov = ts.groupby('month').agg(obs=('id', 'size'), points=('id', 'nunique'))
cov['obs_per_point'] = (cov['obs'] / cov['points']).round(2)
print(cov.to_string())

# Rough guide only: fraction of each month's rows that SCL calls clear (4,5,6).
clear_scl = ts['SCL'].isin([4, 5, 6])
clr = ts.assign(clear=clear_scl).groupby('month')['clear'].mean().round(2)
print("\nSCL-clear fraction by month (guide only — you will apply your own mask):")
print(clr.to_string())

            obs  points  obs_per_point
month                                 
2023-01  511233   74351           6.88
2023-02  505493   74351           6.80
2023-03  508993   74351           6.85
2023-04  505746   74351           6.80
2023-05  501746   74351           6.75
2023-06  503662   74351           6.77
2023-07  508110   74351           6.83
2023-08  507668   74351           6.83
2023-09  512985   74351           6.90
2023-10  598409   74351           8.05
2023-11  506802   74351           6.82
2023-12  427386   74351           5.75
2024-01  504712   74351           6.79
2024-02  504902   74351           6.79
2024-03  514392   74351           6.92
2024-04  513531   74351           6.91
2024-05  501762   74351           6.75
2024-06  506160   74351           6.81
2024-07  505968   74351           6.81
2024-08  592537   74351           7.97
2024-09  507550   74351           6.83
2024-10  514553   74351           6.92
2024-11  427611   74351           5.75
2024-12  588236   74351  

## Reconciliation

In [16]:
returned = set(ts['id'])
if pts is not None:
    recon = pd.DataFrame({
        'input':    pts.groupby('crop').size(),
        'returned': pts[pts['id'].isin(returned)].groupby('crop').size(),
    }).fillna(0).astype(int)
    recon['missing'] = recon['input'] - recon['returned']
    print(recon.sort_values('missing', ascending=False).to_string())
    miss = int(recon['missing'].sum())
    print(f"\nPoints with no observation at all: {miss:,}")
    print("With no masking this should be ~0; any non-zero here means those points fall"
          "\noutside every scene footprint (bad coordinates) — investigate before extraction.")
else:
    print(f"Unique points returned: {len(returned):,} of {n_asset:,} "
          f"(local table not loaded, so no per-crop breakdown)")

            input  returned  missing
crop                                
aman_rice   10798     10798        0
aus_rice     4339      4339        0
banana       4121      4121        0
betel_leaf   2562      2562        0
boro_rice    4241      4241        0
flower       2562      2562        0
groundnut    3643      3643        0
jute         5864      5864        0
maize        4534      4534        0
mustard      2805      2805        0
others       3480      3480        0
pine_apple   4322      4322        0
potato       4145      4145        0
sugarcane    3304      3304        0
tea          3912      3912        0
tobacco      2536      2536        0
vegetables   3131      3131        0
wheat        4052      4052        0

Points with no observation at all: 0
With no masking this should be ~0; any non-zero here means those points fall
outside every scene footprint (bad coordinates) — investigate before extraction.


## Save

In [17]:
desc = ts[SPECTRAL_OUT].describe().round(4).T[['mean', 'min', 'max']]
print(desc.to_string())
print("\nNOTE: because clouds are NOT masked, max values run high and some harmonized DN can")
print("be negative -> negative reflectance after /10000. That is expected in raw data.")

       mean  min     max
B02  0.3152  0.0  3.1752
B03  0.3148  0.0  3.0472
B04  0.2989  0.0  2.6680
B05  0.3406  0.0  2.7218
B06  0.4018  0.0  2.4194
B07  0.4280  0.0  2.2266
B08  0.4299  0.0  2.0072
B8A  0.4395  0.0  2.0200
B11  0.2754  0.0  1.6353
B12  0.2091  0.0  1.6800

NOTE: because clouds are NOT masked, max values run high and some harmonized DN can
be negative -> negative reflectance after /10000. That is expected in raw data.


In [18]:
ts.drop(columns='month').to_parquet(LONG_PARQUET, index=False)
print("Saved:", LONG_PARQUET, f"({os.path.getsize(LONG_PARQUET)/1e6:.1f} MB)")

with open(f'{DIR_QA}/20_s2_extraction_qa.json', 'w') as f:
    json.dump({
        'run_id': RUN_ID, 'generated': pd.Timestamp.now().isoformat(),
        'source': 'COPERNICUS/S2_SR_HARMONIZED (Earth Engine)',
        'config': {'start': START_DATE, 'end': END_DATE,
                   'spectral_bands': SPECTRAL_OUT, 'qa_bands': QA_BANDS,
                   'cloud_masking': 'none (raw)', 'max_scene_cloud': MAX_SCENE_CLOUD,
                   'divide_reflectance': DIVIDE_REFLECTANCE, 'scale_m': SCALE_M,
                   'points_asset': POINTS_ASSET, 'key_field': 'id'},
        'points_asset': int(n_asset), 'points_returned': int(len(returned)),
        'rows': int(len(ts)), 'rows_all_spectral': int(valid.sum()),
        'obs_per_point_median': float(obs.median()),
        'scl_composition_raw': {int(k): float(v) for k, v in comp.items()},
        'monthly_coverage': {str(k): v for k, v in cov.astype(float).to_dict('index').items()},
    }, f, indent=2, default=str)
print("QA saved.")

Saved: /content/drive/MyDrive/Crop_Classification/02_RawTimeSeries/S2/s2raw_202301_202412_long.parquet (239.4 MB)
QA saved.


---
### Next
- Mask consistently (SCL / MSK_CLDPRB) and record the rule — raw extraction just moves the decision downstream.
- Overlapping-tile duplicates (`tile` column): decide once how to resolve, apply uniformly.
- If storage is tight, lower `MAX_SCENE_CLOUD` (~95) rather than re-masking.